## Provenance and scope

This notebook is notebook 2 of the `cbase2/` pipeline. It ports **Steps 4--7**
of the parent `Notebooks/AccountingConsistency.ipynb` (ROADMAP §4.1): the
three-side GDP reconciliation, the model-facing bridge (domestic final demand,
Domar weights, the `consumption_share` calibration), the shock-incidence rule,
artifact emission, and the final validation gate.

**Inputs**: the raw table `data_raw/I-O_DE2019_formatiert.csv` (for the
final-demand columns and sector labels) plus the notebook 01 artifacts in
`data_processed/` (share matrices, value-added decomposition, sector shares,
final-demand taxes). Sector alignment between the two sources is asserted.

**Outputs**: `AC_accounting_reconciliation.csv`, `AC_value_added_components.csv`,
`AC_calibration_table.csv`, `AC_domestic_final_demand.csv`,
`AC_domar_weights.csv` (all under `data_processed/`), and the diagnostic figure
`plots/02_gdp_reconciliation.png`.

**Correction vs the parent's last version**: the parent's validation section
described the production-vs-expenditure residual as "< 1.5%" while the actual
documented value is $\approx 5.4\%$ and the hard gate is 10%. The text here
states the real figure; the residual is a raw-table valuation discrepancy to be
documented, not fixed.


## Step 0 -- Load the raw table and the notebook 01 artifacts

The raw table is re-read (final-demand columns and sector labels); everything
model-facing that Step 1 built arrives as CSV artifacts and is checked for
sector alignment against the raw table before any computation.


In [ ]:
import Pkg

# Walk upward from this notebook to the nearest Project.toml (the BeyondHulten
# project root). Skipped when the running environment already provides the
# packages (e.g. the container stack environment used for headless validation).
function find_project(start)
    d = abspath(start)
    while d != dirname(d)
        isfile(joinpath(d, "Project.toml")) && return d
        d = dirname(d)
    end
    return nothing
end
if Base.find_package("CSV") === nothing
    Pkg.activate(find_project(@__DIR__))
end
using CSV, DataFrames, LinearAlgebra, Statistics, Plots

CBROOT = abspath(@__DIR__)
RAW    = joinpath(CBROOT, "data_raw", "I-O_DE2019_formatiert.csv")
OUT    = joinpath(CBROOT, "data_processed")
PLOTS  = joinpath(CBROOT, "plots")

# Same schema constants as notebook 01 (kept in sync by assertion below).
N   = 71
SEC = 2:72
FD  = 75:81     # private cons, priv-orgs cons, gov cons, equipment inv,
                # construction inv, inventories, exports

io = CSV.read(RAW, DataFrame; delim=';', decimal=',', missingstring=["-", "x"])
rename!(io, Symbol(names(io)[1]) => :Sektoren)
io.Sektoren = replace.(io.Sektoren, r"^\s+" => "")
io = coalesce.(io, 0)

# --- Notebook 01 artifacts ---
va     = CSV.read(joinpath(OUT, "wrangling_value_added.csv"), DataFrame)
sh     = CSV.read(joinpath(OUT, "wrangling_sector_shares.csv"), DataFrame)
imptx  = CSV.read(joinpath(OUT, "wrangling_final_demand_taxes.csv"), DataFrame)
omega_raw_df = CSV.read(joinpath(OUT, "wrangling_omega_raw.csv"), DataFrame)
Ω_raw  = Matrix{Float64}(omega_raw_df[:, 2:end])     # drop the user-label column

# --- Alignment checks: artifacts must refer to the same sectors in order ---
@assert collect(va.sector) == collect(io.Sektoren[1:N]) "VA artifact sector mismatch"
@assert collect(sh.sector) == collect(io.Sektoren[1:N]) "share artifact sector mismatch"
@assert size(Ω_raw) == (N, N) && all(isapprox.(vec(sum(Ω_raw; dims=2)), 1.0; rtol=1e-9) .| (vec(sum(Ω_raw; dims=2)) .<= 0.0)) "Ω_raw rows must sum to 1 (or 0)"
@assert nrow(imptx) == length(FD) "final-demand tax artifact must have 7 categories"

gva, wage, othertx, dep, netop = va.gross_value_added, va.wage,
    va.other_production_tax, va.depreciation, va.net_operating_surplus
prodval, factor_share = sh.gross_output_basic, sh.factor_share
imp_final, imptx_final = imptx.imported_final, imptx.product_tax_final

println("Loaded raw table + 5 notebook 01 artifacts; sector alignment OK.")


## Step 4 -- Reconcile the three GDP sides

All three measures at **basic prices**:

- **Production:** $\sum_s \text{gross value added}_s$
- **Income:** $\sum_s (\text{wage} + \text{other tax} + \text{dep} + \text{net operating surplus})_s$
- **Expenditure:** domestic final demand at basic prices
  $= \sum_c (\text{final demand}_c - \text{imported final}_c - \text{product tax on final}_c)$
  summed over the seven final-demand categories **including exports** (the
  parent pipeline's documented convention).

Production and income are identical by construction. Expenditure deviates by
the documented raw-table statistical discrepancy ($\approx 5.4\%$), which arises
from (a) the basic/purchaser product-tax bridge (product taxes appear on both
intermediate and final uses) and (b) the proportional import allocation of
notebook 01, Step 1. We **document, not hide** it.


In [ ]:
# ---------------------------------------------------------------------------
# Step 4 -- Reconcile the three GDP measures (basic prices).
# ---------------------------------------------------------------------------
GDP_P = sum(gva)                                            # production (basic)
GDP_I = sum(wage) + sum(othertx) + sum(dep) + sum(netop)    # income (basic)
fd_purch_total = sum(Matrix(io[1:N, FD]))                   # final demand, purchaser, dom+imp
fd_dom_basic   = fd_purch_total - sum(imp_final) - sum(imptx_final)
GDP_E          = fd_dom_basic                               # expenditure (basic)

println("=== GDP reconciliation (basic prices, EUR m) ===")
println("GDP production : ", round(GDP_P))
println("GDP income     : ", round(GDP_I))
println("GDP expenditure: ", round(GDP_E))
println("|P - I| = ", round(abs(GDP_P - GDP_I); digits=2))
println("|P - E| = ", round(abs(GDP_P - GDP_E); digits=2), "  (",
        round(abs(GDP_P - GDP_E)/GDP_P*100; digits=3), "% -- documented raw-table discrepancy)")
println("|I - E| = ", round(abs(GDP_I - GDP_E); digits=2))


## Model-facing bridge -- domestic final demand, Domar weights, consumption calibration

The vectors the CGE actually consumes, built only from domestic, basic-price
quantities:

- `fd_dom_basic_vec[s]` -- domestic final demand by sector (basic), with
  imports and product taxes stripped proportionally per final-demand category;
  negative domestic fractions (imports + taxes exceeding a category's total)
  are clamped to zero and the clamp is **reported**, not silent.
- $\lambda_s = Y_s / \text{GDP}$ -- the **standard Domar weights** (always
  positive; the sum is the Domar aggregate $> 1$).
- `cons_share` -- the Beyond-Hulten `consumption_share` calibration, computed
  from the **total** conditional shares $\Omega^{raw}$ (not the domestic audit
  matrix) as $(I - \mathrm{diag}(1 - \text{factor\_share})\,\Omega^{raw})' \, Y$.
- `labor_share_model` -- the model's historical "labour share", i.e. the
  composite value-added weight $\lambda_s \cdot$ `factor_share`.


In [ ]:
# ---------------------------------------------------------------------------
# Bridge: fd_dom by sector/category, cons_share calibration (Ω_raw), λ, labor_share.
# ---------------------------------------------------------------------------
fd_bysector = Matrix{Float64}(io[1:N, FD])          # purchaser, dom+imp, sector x category
fd_dom_basic_bysector = similar(fd_bysector)
for (k, c) in enumerate(FD)
    cat_total = sum(fd_bysector[:, k])
    imp_c = imp_final[k]; tx_c = imptx_final[k]
    # Domestic fraction of this final-demand category after removing imports + taxes.
    domfrac = cat_total > 0 ? (cat_total - imp_c - tx_c) / cat_total : 0.0
    domfrac < 0 && println("WARNING: negative domestic fraction (", round(domfrac; digits=4),
                           ") clamped to 0 in final-demand category ", k)
    fd_dom_basic_bysector[:, k] = fd_bysector[:, k] .* max(domfrac, 0.0)
end
fd_dom_basic_vec = vec(sum(fd_dom_basic_bysector; dims=2))

grossy = prodval                                     # gross output at basic prices
# Beyond-Hulten "consumption_share" (value of final demand absorbed per unit gross output).
cons_vec   = (I - Diagonal(1.0 .- factor_share) * Ω_raw)' * grossy
cons_vec   = max.(cons_vec, 0.0)
cons_share = cons_vec / sum(cons_vec)
λ = prodval ./ GDP_P                                 # standard Domar weights
labor_share_model = λ .* factor_share                # composite VA weight

# Validation ABSENT from the parent notebook's bridge:
@assert all(isfinite, cons_vec) && all(cons_vec .>= 0) "cons_vec must be finite and non-negative"
@assert isapprox(sum(cons_share), 1.0; rtol=1e-9)      "cons_share must sum to 1"
@assert all(isapprox.(vec(sum(Ω_raw; dims=2)), 1.0; rtol=1e-9) .| (vec(sum(Ω_raw; dims=2)) .<= 0.0)) "Ω_raw rows sum to 1 (or 0) (re-checked)"

println("Domar weights λ: sum = ", round(sum(λ); digits=4),
        "  min = ", round(minimum(λ); digits=4), "  max = ", round(maximum(λ); digits=4))
println("cons_share: sum = ", round(sum(cons_share); digits=9),
        "  min = ", round(minimum(cons_share); digits=6),
        "  max = ", round(maximum(cons_share); digits=6))


## Domar weights -- assertion cell

The **standard Domar weight** for sector $s$ is
$\lambda_s^{Domar} = Y_s / \text{GDP}$ with $Y_s$ gross output at basic prices
and GDP the sum of gross value added. Weights are always positive and finite;
the aggregate $\sum_s \lambda_s > 1$ because gross outputs double-count
intermediates. The bridge cell above already uses this definition; this cell
turns the equality into explicit assertions.


In [ ]:
# ---------------------------------------------------------------------------
# Assert the Domar weights: standard definition λ_s = Y_s / GDP.
# ---------------------------------------------------------------------------
GDP_for_domar = GDP_P
λ_domar = prodval ./ GDP_for_domar

@assert all(isfinite, λ_domar) "Domar weights must be finite"
@assert all(λ_domar .> 0)      "standard Domar weights must be positive (λ_s = Y_s/GDP)"
@assert λ_domar ≈ λ            "bridge Domar weights must equal the standard definition"
@assert sum(λ_domar) > 1       "Domar aggregate (sum of gross outputs / GDP) should exceed 1"

println("Domar weights (standard λ = Y/GDP):")
println("  all positive? ", all(λ_domar .> 0), "  all finite? ", all(isfinite, λ_domar))
println("  sum(λ) = ", round(sum(λ_domar); digits=4),
        "  (> 1: intermediates double-counted)")


## Step 5 -- Shock incidence rule (§4.1)

The Hornykewycz (2025) shock is the investment impulse in
`data_raw/impulses.csv`. §4.1 requires us to **state whether the shock hits
domestic or imported demand**:

> The shock is applied to **domestic final demand only**; imported content is
> held fixed. The target vector is `fd_dom_basic_vec` built above.

Under the cbase2 financing closures this rule is refined further (see
`03_financing_closures.ipynb`, once built): the unfinanced autonomous shock is
retired, and every experiment closes through F1 (preference reallocation), F2
(tax-financed $g_i$ with $\sum_i p_i g_i = T$), or F3 (external debt). The
impulse file is loaded defensively here so the rule stays checkable against
the paper's stated $\approx$ EUR 40.3 bn at 2019 prices.


In [ ]:
# ---------------------------------------------------------------------------
# Step 5 -- Shock incidence rule (documentation + defensive load).
# ---------------------------------------------------------------------------
println("=== Shock incidence (§4.1) ===")
println("Rule: the Hornykewycz (2025) investment shock is applied to DOMESTIC final")
println("demand only; imported content is held fixed. Target vector = fd_dom_basic_vec.")
println("(cbase2 refinement: every matrix cell is financed via F1/F2/F3; the")
println(" unfinanced autonomous shock is retired -- see 03_financing_closures.ipynb.)")
try
    imp = CSV.read(joinpath(CBROOT, "data_raw", "impulses.csv"), DataFrame)
    numcols = [c for c in propertynames(imp) if eltype(imp[!, c]) <: Real && c != :year]
    shock_total = sum(skipmissing(imp[1, c] for c in numcols))
    println("impulses.csv: rows=", nrow(imp), " cols=", ncol(imp))
    println("Aggregate nominal shock (row 1, sum of numeric cols) = ", round(shock_total))
    println("(Paper states ≈ EUR 40.3 bn at 2019 prices; reconcile against this figure.)")
catch e
    println("Could not auto-load impulses.csv (", e, "); rule documented above stands.")
end


## Step 6 -- Emit calibration artifacts (CSVs)

All model-facing artifacts are written to `data_processed/` (the parent wrote
them to its shared `output/`; cbase2 keeps pipeline outputs in the pipeline):

- `AC_accounting_reconciliation.csv` -- the three GDP measures and pairwise differences.
- `AC_value_added_components.csv` -- the four value-added components by sector.
- `AC_calibration_table.csv` -- per-sector calibration (gross output, VA
  components, factor share, wage share, import share, Domar weight, domestic
  final demand).
- `AC_domestic_final_demand.csv` -- domestic final demand by category (basic);
  `AC_domestic_intermediate_matrix.csv` was already emitted by notebook 01.
- `AC_domar_weights.csv` -- sector Domar weights.


In [ ]:
imp_inter_byuser = CSV.read(joinpath(OUT, "wrangling_imports_taxes.csv"), DataFrame).imported_intermediate
# ---------------------------------------------------------------------------
# Step 6 -- Persist artifacts (CSVs) for transparency + downstream model use.
# ---------------------------------------------------------------------------
CSV.write(joinpath(OUT, "AC_accounting_reconciliation.csv"),
    DataFrame(measure=["GDP_production","GDP_income","GDP_expenditure","abs_P_I","abs_P_E","abs_I_E"],
              value=[GDP_P, GDP_I, GDP_E, abs(GDP_P-GDP_I), abs(GDP_P-GDP_E), abs(GDP_I-GDP_E)]))
va_comp2 = DataFrame(sector = va.sector, wage = wage, other_production_tax = othertx,
                     depreciation = dep, net_operating_surplus = netop,
                     gross_value_added = gva)
CSV.write(joinpath(OUT, "AC_value_added_components.csv"), va_comp2)

calib = DataFrame(sector = io.Sektoren[1:N],
    gross_output_basic = prodval,
    gross_value_added = gva,
    wage = wage, other_production_tax = othertx, depreciation = dep, net_operating_surplus = netop,
    factor_share = factor_share, wage_share_gross_output = wage ./ prodval,
    imported_intermediate = imp_inter_byuser,
    import_share = imp_inter_byuser ./ max.(prodval, 1e-12),
    domar_lambda = λ_domar,
    final_demand_domestic_basic = fd_dom_basic_vec)
CSV.write(joinpath(OUT, "AC_calibration_table.csv"), calib)

CSV.write(joinpath(OUT, "AC_domestic_final_demand.csv"),
    DataFrame(sector = io.Sektoren[1:N],
              private_consumption = fd_dom_basic_bysector[:,1],
              private_orgs_consumption = fd_dom_basic_bysector[:,2],
              government_consumption = fd_dom_basic_bysector[:,3],
              equipment_investment = fd_dom_basic_bysector[:,4],
              construction_investment = fd_dom_basic_bysector[:,5],
              inventories = fd_dom_basic_bysector[:,6],
              exports = fd_dom_basic_bysector[:,7]))
CSV.write(joinpath(OUT, "AC_domar_weights.csv"), DataFrame(sector = io.Sektoren[1:N], lambda = λ_domar))

println("Artifacts written to ", OUT)
for f in ["AC_accounting_reconciliation.csv","AC_calibration_table.csv",
          "AC_value_added_components.csv","AC_domestic_final_demand.csv",
          "AC_domar_weights.csv"]
    @assert isfile(joinpath(OUT, f)) "missing artifact: $f"
    println("  wrote data_processed/", f)
end
println("(AC_domestic_intermediate_matrix.csv was emitted by notebook 01.)")


## Diagnostic figure -- GDP reconciliation

The three GDP sides side by side. The visual gap between production/income and
expenditure is the documented $\approx 5.4\%$ raw-table discrepancy -- a feature
of the source data at this valuation, not a pipeline error.


In [ ]:
# ---------------------------------------------------------------------------
# Diagnostic: three-side GDP reconciliation bar chart.
# ---------------------------------------------------------------------------
p = bar(["Production", "Income", "Expenditure"], [GDP_P, GDP_I, GDP_E];
        label = "", title = "GDP three-side reconciliation (basic prices, EUR m)",
        ylabel = "EUR m", ylims = (0, maximum([GDP_P, GDP_I, GDP_E]) * 1.12),
        size = (700, 420), margin = 4Plots.mm)
savefig(p, joinpath(PLOTS, "02_gdp_reconciliation.png"))
display(p)
println("Saved plots/02_gdp_reconciliation.png")


## Step 7 -- Validation and reconciliation assertions

Final guardrails. Every assertion below must hold for the accounting to be
considered consistent. The only tolerated deviation is the documented GDP
production-vs-expenditure residual, whose actual magnitude in this data is
$\approx 5.4\%$ (raw-table statistical discrepancy; see the parent repo's
accounting-consistency plan). The hard gate rejects a **gross** error
($\geq 10\%$); the residual itself is expected and documented, not "fixed".


In [ ]:
# ---------------------------------------------------------------------------
# Step 7 -- Validation and reconciliation assertions.
# ---------------------------------------------------------------------------
println("=== Validation ===")
@assert isapprox(wage .+ othertx .+ dep .+ netop, gva; rtol=1e-9) "VA decomposition must equal GVA"
@assert GDP_P ≈ GDP_I "production must equal income"
resid = abs(GDP_P - GDP_E) / GDP_P
println("GDP production vs expenditure residual = ", round(resid*100; digits=3),
        "% (raw-table statistical discrepancy; documented, not fixed)")
@assert resid < 0.10 "production vs expenditure discrepancy >= 10% (gross accounting error)"
@assert all(x -> x >= 0, calib.gross_value_added) "value added must be non-negative"
@assert all(isfinite, λ) "Domar weights must be finite"
@assert all(λ .> 0)      "standard Domar weights must be positive (λ_s = Y_s/GDP)"
@assert sum(λ) > 0       "sum of Domar weights must be positive"
println("All assertions passed.")
println("Domar weights λ: all positive? ", all(λ .> 0), "  all finite? ", all(isfinite, λ),
        "  sum = ", round(sum(λ); digits=4), " (> 1: Domar aggregate)")
